## CASH Notebook

## Celestial Chase - LVL 1: The Teal Beacon

You've just woken up from cryo-sleep. No memory. No crew. Just you, a dying sun, and a faint signal pulsing from the sky.

One star glows different from the rest. Not white. Not grey. **Teal.**

Find it. Its position holds part of the key. But position alone is not enough - you must also know how much of its face is lit by the sun.

---

**The encoded signal:** `toecbj`

**Your task:**
1. Display the image and find the **cyan pixel** - it is the only pixel where `B == 255` and `G == 255` and `R == 0`
2. Read its `(x, y)` coordinates
3. Use `ephem` to compute **Venus's phase** (`int(venus.phase)`) on `2025/6/21 12:00:00` UTC from Zurich (lat=`47.3769`, lon=`8.5417`)
4. Build the key: `key = x * y + int(venus.phase)`
5. Build the permutation: `random.Random(key).shuffle(list(range(len(encoded))))`
6. Reverse the transposition to decode: `decoded[i] = encoded[perm[i]]`


In [ ]:
import base64, cv2, numpy as np
from IPython.display import display, Image as IPImage

# Starfield image embedded directly in this notebook
img_b64 = "iVBORw0KGgoAAAANSUhEUgAAAZAAAAGQCAIAAAAP3aGbAAAOc0lEQVR4Ae3Be7zPBZ7H8ff7/53H7D6maTK6WFKji6aLrXbKtcRBiBpEFCGHg5NrLhmX3A8OR4giwogQjku51my1uky6GIlVMpqmeezOY/f/s49H+9jHox5F5zi/y/fz+76eTwsAgrCQAPte+UPLu38jAOdlAUAQFhLsmSXPPzrwIQH4hlWIPj12+sqrLxOAwmIBQBAWkGwlxSPLK2YLkCxkzn0du7+0dZ0AZIcFIK1WLFvTt39PxWEBiKZj+we2bt+g9LG+67H+w55eNl8AkDwWCs7J42fqN6wroOBYABCEBQBBWAAQhAUAQVj4rsULVwwa0ldATSwoWzK0dKCQZRYABGGhJra+tKvjfW0EIB8sAAjCApB9xQNLK5aUCbVjAUiT8WOnTJ0+QTFZADJhQL+SpcvLhWyywpo2ec64iSMEpNvG32/r+tsOSgcLAIKwACAIK62Gl4yZVz5DAOKwgBR47cC/39n8X4TgLAAIwgKAICwACMICgCAsAAjCApA5o0dMnDlnspAdFvBduysP3FPUXEDyWEA1LJy/bMiw/gLyysIFadms7b6DO1VwHun92LOrnhaQSBaSbfPGys5diwRAspA5pUPHli2YLgDZYQFAEFZO9O0zaMXKxUJuTXhi6pSnxgsoFBYABGHV0Htvf3zjLdcIAHLOAhDEl2f+85K6/6QUswAgCAtIh7LZFaUji4XILAAIwqqFHS+/2u7euwQA2Xdfx+4WAARh1VCfXgNWrl4q1EK/h4uXP1chADVkAUAQFgAEYQFAEBYABGEBQBAWAHzLh+8fv+6GhkokCwCCsADgh6xYtqZv/55KEgsAgrCQZW3u7rjrla0CUGsWgDjGj50ydfoEpZUFII6RpeNnl01VWlkAEIQFAEFYQAL8Q1XV/9gCzss6r149+q1eu1wAkG+7Kw9YABJg7KhJ02dNEs7LAs7rs5NfXlH/EgEJYAFAEBYABGEBQBAWAARhAUCC9ezed826FfqGBQBBWAAQhFVDM6bNGzNuuAAg5ywACMICgCAsAAjCAoAgLAAIwgpo6u9mj39ypACkjAUAQVgAEIQFAEFYQDaNenzCrLlTBGSCBQBBWAAQhAUAQVgAEIQFAEFY+fOzqqq/2QJQa2c+/7ru5Rcp2da/sLnbg51VCxYABGEBQBAWAARhIfFmPjV/9BPDBKSeBQBBWAAQhAWgsLRs1nbfwZ0qRBYABGEBQBAWAARhAUAQVu0U3dO5cvdm1dyxj09dfU09ZUHrVh327N0mAHl14pMvGlx1qTLKAoAgLAAIwgIyqkNR122VGwVkgQUAQVgAMuTIe8ca33i1kDVWynx05NNrG18pAAFZABCEBQBBWAAQhAUAQVgAkG8V5cuLS/rpx1hACkyZNGvCpFFCcBYABGHVXEnxyPKK2QKA3LISb0nFcwOLHxaQK6NHTJw5Z7KQPBaATOjZve+adSuEbLIAIAgLAIKwgHwbN2bytBkTBfwYC0DtVVXJVkEb+OjQJc8sUJ7MnbXo8VGDLQC1VFWl/2ML2WQBqL2qKtlCllkAUDt9eg1YuXqpss8CgCAsXJA/ffQfv7r2nwUgh6x8e+H5jQ8+1FUA8GMsADlXUb68uKSfkmH50tX9BvRSBBYABGEBuCCTJkyfNGWskENW5qx/YXO3BzsLwPcsXrhi0JC+Qu1YAKKZNnnOuIkjlD4WEMrQwaMWLJolpJIFAEFYABCEBQCJt/WlXR3va2MByJyjH55sdF19ITssAAjCAoAgLAAIwgKAICwAOdelU49NW9YKNWQBQBAWACTY2tWbevTqom9YP6ZV86K9ByqVSK2aF+09UCkA6WABQBAWAARhAUAQFgAEYQFAEBYABGEBQBAWAARhAVnTqnnR3gOVAjLEQv68suvQ3W2aCsi5dw9/dFOTaxWNlSG9e/ZftWaZACBrLAAIwgKAICwEdPTDk42uqy8gZSwACMICgCAsAAjCSoH2bbts37lJAIKzACAICwCCsAAgCAtAlh3c92azlrepcC1euGLQkL7KPgsoCMNLxswrnyEUNAuF5ch7xxrfeLWAQmQBQBAWAARhIb6D+95s1vI21cTTi559bPAjAkKxgAK1Z+fB1m2bCQXEQm6tWfViz973C0DNWQAQhAUAQVjIhOElY+aVzxC+5/CbR5rc1lgIpU+vAStXL1XyWAAQhAUAQVgAEIQFAEFYCfPUlLlPTHhc57B65YZefR4QgFSyUEN3t2z/yr7tArLslV2H7m7TVPgWC8iEN15/9/Y7bhKQTRYABGEBQBAWAARhAciJzRsrO3ct0nd9+P7x625oKFSPVShe3f3aXffcKQCFywLy5/WDh+9o1kRA9VgAEIQFAEFYABCEBQBBWAAQhFXQjn54stF19QWgIFgAkuGhBx99/oVnlFQTx02bPG2c8soCgCAsAAjCwvf88Z2jv765kYBaa9W8aO+BSiFDrDSpW1V1xhaAmCwACMJCdnTt/ODGzS8IQOZYAPBDPj12+sqrL1OSWAAQhAUAQVgAkAnPLV/7cL8eyiYrT7a+tKvjfW0EANVmZcjI0vGzy6YKALLGAoAgLAB51aGo67bKjUI1WAAQhAUAQVgAInj38Ec3NblWSTJl0qwJk0Yphyyc2+wZ5SPHlAhAMlgALsiuHfvbtGsh5JCFNFlQtmRo6UABMVkAEIQFAEFYyITfTZzx5OQxApBNFgAEYQEF7SdVVf9tC3lyb7v7X97xojLEAoAgLADfKB5YWrGkTEgwCwCCsAAgCAsAgrAAIAgL5/bJ0c+uanSFACSDBSDn3j380U1NrhVqyAKAICwASLaPjnx6beMrJVlA6j296NnHBj8iZEJF+fLikn7KDguonjkzF44YPURA/lgAEIQFAEFYABCEhbCGDh61YNEsAalhAUAQVoqd+fzrupdfJABBWCmwa8f+Nu1aqHC98fq7t99xk1DoTp/66rJ6F+tCHdr/VtMWtyoyC0DtdGz/wNbtG1Q9O7fva9u+pXBBrGo7efxM/YZ1BQB5YkXw1dm/X1znp0IKdOnUY9OWtQJ+iAUAQVgIZf0Lm7s92FlAKllAsp06cbZegzoCJAsAgrBi+m3Xh36/8XllyJnPv657+UUCkGwWAARhAUAQFoDgigeWViwpUwpYCOKzk19eUf8SIRO6P9Bn3YaVQjQWAARhAUD+DBowfPHSeaoeCwCCsBJmx8uvtrv3LgFxfPHZXy+94udC9lkAEIQFBDRowPDFS+cpgm1b9nTo1FrIBAsR3NWi3av7dwhINwsAgrAAIAgrGdq1uW/HrpcE/JAzn39d9/KLFNy/vfbOv955s1ALFgAEYSGyg/vebNbyNgHpYIVy6sTZeg3qCEBSHdz3ZrOWtyk7LKTSZVVVp20BoVgAEIQFAEFYABCEBQBBWAAQhIX4Th4/U79hXQGFzgKAICwACMIC0u3UibP1GtQRvuepKXOfmPC4ksQCgCAsAAjCiuzRRwY/8+wiVU+3+3uvf3GVAIRlIZFmzygfOaZEAL7FAoAgLAAIwgKQSB9/cOKa6xsI32IBQBAWAFTP9KllY8eXKn8s1NzKFev69O0uAOe16tn1vR/ppsyxgHx4rP+wp5fNF1ATFgAEYQFAEBYABGEBQBAWAARhAUAQFpBNRz882ei6+gIywcJ5vX7w8B3NmghAAlhArnzx2V8vveLnAi6UBeBblj29qv9jvXVec2YuHDF6iJBzFhLp0P63mra4VQlWUb68uKSfgByyACAICwCq5+MPTlxzfQPljwUAQVgAEISFanj7rQ9uufV6AcgrCwCCsBDKyeNn6jesKyDf+vcdsmzFQuWWhULX+d5um19eLyA+C0iSM59/XffyiwT8EAsAgrAAVNvDDw187vklQp5YSJ46VVVnbWXBPXfdu/vVlwXEZAHAuU1+cubE341WMlgpM2zI6PkLZwpAQBai6dj+ga3bNyhlOrZ/YOv2DUK6WUCmjRg+bs68aQIyzQKAIKwaOvHJFw2uulQAkHMWAARhfcuWTTs7dWkrAEgkCwCCsAAgCAsAgrCAJFm04JnBQx9VDp0+9dVl9S4WIrCAaiu6p3Pl7s2lQ8eWLZguIOcsAAjCAoAgrIQ5deJsvQZ1BADfYwFAEBYABGEBQBAWgOQ5/OaRJrc1Fr7LAoAgLADIsvfe/vjGW65RrVkAEIQFAEFYABCEBZxXm7s77nplq4AEsAB8o1Xzor0HKoUEswAgCAsAgrAAIAgLSIYunXps2rJWwLlZefXZyS+vqH+JAKAaLAAIwgKAICwACMICgFz58+m//fKyn+lCWQAQhAUAQVhSt/t7r39xlZAyK5at6du/p4A4LAAIwkLB6dm975p1KwQUHAtAtb3/7p9uuOlXKixfnf37xXV+qmpr27rTzj1blA8WAHxj2JDR8xfOVIJZABCEBaTArOkLRo0dKgRnAUAQVjSDBgxfvHSeAKSPlXqtW3XYs3ebgDRZ9ez63o90UzQWgFDmzlr0+KjBSiULAIKw8qd1qw579m4TkGVH3jvW+MarhfgsAAjCAoAgLGTfof1vNW1xqwDUjgUAQVgAqmHvntdbtb5DyCsLAIKwACAICwCCsAAgCAuI44/vHP31zY2EtLIAIAgLAIKwACAICwCCsKrtF1VVf7H1Y2bPKB85pkQAamjKpFkTJo0Szs0CQmnftsv2nZuEVLKS4cDeN5q3ul0AcG4WqqekeGR5xWwByB8rxaZPLRs7vlQAgrAAIAgLOLe//Pm/fvHLfxSSquiezpW7Nys1LACpVz5vacnwAUo8CwBq6OMPTlxzfQPlnAUAQVgAEISFlBk2ZPT8hTMFBGQBQBAWAARhAUDWjHp8wqy5U5QhVlId2v9W0xa3CgD+nwUAQVioieKBpRVLygQgHywACMICgCAsFLrSoWPLFkwXEJ+FDNm2ZU+HTq1VcP5w6O3fNL1FQAL8L8zJ9dhllQWVAAAAAElFTkSuQmCC"

img_bytes = base64.b64decode(img_b64)
img_arr   = np.frombuffer(img_bytes, dtype=np.uint8)
img       = cv2.imdecode(img_arr, cv2.IMREAD_COLOR)

cv2.imwrite('starfield.png', img)
display(IPImage('starfield.png'))
print("Image shape:", img.shape)


In [ ]:
import ephem, random

encoded  = "toecbj"
obs_date = "2025/6/21 12:00:00"
obs_lat  = "47.3769"
obs_lon  = "8.5417"

# TODO Step 1: Find the cyan pixel (B=255, G=255, R=0) in img
# Hint: use np.where or a loop
pixel_x, pixel_y = 0, 0  # replace with actual coordinates

# TODO Step 2: Compute Venus phase with ephem
obs = ephem.Observer()
obs.lat  = obs_lat
obs.lon  = obs_lon
obs.date = obs_date
venus = ephem.Venus()
venus.compute(obs)
phase = 0  # replace: int(venus.phase)

# TODO Step 3: Build key and permutation
key  = pixel_x * pixel_y + phase
perm = list(range(len(encoded)))
random.Random(key).shuffle(perm)

# TODO Step 4: Reverse the transposition
# Hint: if encoded[perm[i]] = original[i], then decoded[i] = encoded[perm[i]]? Think carefully.
def transpose_decode(encoded, perm):
    pass  # implement this

answer = transpose_decode(encoded, perm)
print(answer)
